In [19]:
from google.colab import auth
auth.authenticate_user()
from google.cloud import bigquery

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

print('Authenticated')

Authenticated


In [20]:
!pip install google-cloud

In [21]:
#підключення до БЗ bigquery

PROJECT_ID = "data-analytics-mate"

client = bigquery.Client(project=PROJECT_ID)

In [22]:
#запит до БД для формування датасету на основі таблиць

query = """
    SELECT
    s.date as date,
    s.ga_session_id as session_id,
    sp.continent,
    sp.country,
    sp.device,
    sp.browser,
    sp.mobile_model_name,
    sp.operating_system,
    sp.language,
    sp.medium as traffic_source,
    sp.channel,
    acs.account_id,
    ac.is_verified,
    ac.is_unsubscribed,
    p.category,
    p.name as product_name,
    p.price,
    p.short_description as description
    FROM `DA.session` as s
    LEFT JOIN `DA.session_params` as sp
    ON s.ga_session_id = sp.ga_session_id
    LEFT JOIN `DA.account_session` as acs
    ON s.ga_session_id = acs.ga_session_id
    LEFT JOIN `DA.account` as ac
    ON acs.account_id = ac.id
    LEFT JOIN `DA.order` as o
    ON s.ga_session_id = o.ga_session_id
    LEFT JOIN `DA.product` as p
    ON o.item_id = p.item_id
"""

df = client.query(query).to_dataframe()
df.head()

,date,session_id,continent,country,device,browser,mobile_model_name,operating_system,language,traffic_source,channel,account_id,is_verified,is_unsubscribed,category,product_name,price,description
0,2020-11-01,5760483956,Americas,United States,desktop,Chrome,Safari,Macintosh,zh,<Other>,Paid Search,<NA>,<NA>,<NA>,Bookcases & shelving units,VITTSJÖ,609.0,"Shelving unit with laptop table, 202x36x175 cm"
1,2020-11-01,7115337200,Europe,United Kingdom,desktop,Chrome,Chrome,Web,en-us,organic,Organic Search,<NA>,<NA>,<NA>,Bookcases & shelving units,VITTSJÖ,609.0,"Shelving unit with laptop table, 202x36x175 cm"
2,2020-11-01,3978035233,Europe,Norway,mobile,Chrome,<Other>,Web,zh,(none),Direct,<NA>,<NA>,<NA>,Tables & desks,RÅSKOG,189.0,"Trolley, 35x45x78 cm"
3,2020-11-01,9648986282,Africa,Nigeria,mobile,Chrome,<Other>,Android,es-es,(none),Direct,<NA>,<NA>,<NA>,Bookcases & shelving units,VITTSJÖ,609.0,"Shelving unit with laptop table, 202x36x175 cm"
4,2020-11-01,4393441533,Asia,China,desktop,Chrome,Chrome,Windows,en-us,(none),Direct,<NA>,<NA>,<NA>,Bookcases & shelving units,VITTSJÖ,609.0,"Shelving unit with laptop table, 202x36x175 cm"


1. ### ***Data overview. Розуміння даних та їх змісту***

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 349545 entries, 0 to 349544
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   date               349545 non-null  dbdate 
 1   session_id         349545 non-null  Int64  
 2   continent          349545 non-null  object 
 3   country            349545 non-null  object 
 4   device             349545 non-null  object 
 5   browser            349545 non-null  object 
 6   mobile_model_name  349545 non-null  object 
 7   operating_system   349545 non-null  object 
 8   language           235279 non-null  object 
 9   traffic_source     349545 non-null  object 
 10  channel            349545 non-null  object 
 11  account_id         27945 non-null   Int64  
 12  is_verified        27945 non-null   Int64  
 13  is_unsubscribed    27945 non-null   Int64  
 14  category           33538 non-null   object 
 15  product_name       33538 non-null   object 
 16  pr

In [24]:
df.describe()

,session_id,account_id,is_verified,is_unsubscribed,price
count,349545.0,27945.0,27945.0,27945.0,33538.000000
mean,4992250296.631739,659005.065557,0.71698,0.16944,953.298679
std,2887450949.537772,13216.529465,0.450474,0.375147,1317.001775
min,1205.0,636133.0,0.0,0.0,3.000000
25%,2493646855.0,647576.0,0.0,0.0,170.000000
50%,4988476074.0,658952.0,1.0,0.0,445.000000
75%,7491286508.0,670414.0,1.0,0.0,1195.000000
max,9999997129.0,681962.0,1.0,1.0,9585.000000


✅ ***Загальна кількість колонок***

In [25]:
print('загальна кількість колонок:', df.shape[1])

загальна кількість колонок: 18


✅ ***Кількість та назва колонок числового типу***

In [26]:
numeric_df = df.select_dtypes(include=['number'])
print('кількість колонок числового типу:',  numeric_df.shape[1])
print('які саме колонки числового типу:',  numeric_df.columns.to_list())

кількість колонок числового типу: 5
які саме колонки числового типу: ['session_id', 'account_id', 'is_verified', 'is_unsubscribed', 'price']


✅ ***Кількість та назва колонок категоріального типу***

In [27]:
categ_df = df.select_dtypes(include=['object', 'category'])
print('кількість колонок категоріального типу:',  categ_df.shape[1])
print('які саме колонки категоріального типу:',  categ_df.columns.tolist())

кількість колонок категоріального типу: 12
які саме колонки категоріального типу: ['continent', 'country', 'device', 'browser', 'mobile_model_name', 'operating_system', 'language', 'traffic_source', 'channel', 'category', 'product_name', 'description']


✅ ***Кількість колонок типу datetime***

In [28]:
datetime_df = df.select_dtypes(include=['dbdate', 'datetime'])
print('кількість колонок числового типу:',  datetime_df.shape[1])
print('які саме колонки числового типу:',  datetime_df.columns.to_list())

кількість колонок числового типу: 1
які саме колонки числового типу: ['date']


✅ ***Кількість унікальних сесій***

In [29]:
print('кількість унікальних сесій:', len(pd.unique(df['session_id'])))

кількість унікальних сесій: 349545


✅ ***Період часу***

In [31]:
print(f'період часу розглядається з {min(df['date'])} до {max(df['date'])}')

період часу розглядається з 2020-11-01 до 2021-01-31.


✅ ***Наявність пропущених значень та їх доля***

In [32]:
missing_values = df.isna().sum()
print("Пропущені значення:", missing_values)

missing_percent  = df.isna().mean() * 100
print("\nДоля пропущених значення:", round(missing_percent , 1))

Пропущені значення: date                      0
session_id                0
continent                 0
country                   0
device                    0
browser                   0
mobile_model_name         0
operating_system          0
language             114266
traffic_source            0
channel                   0
account_id           321600
is_verified          321600
is_unsubscribed      321600
category             316007
product_name         316007
price                316007
description          316007
dtype: int64

Доля пропущених значення: date                  0.0
session_id            0.0
continent             0.0
country               0.0
device                0.0
browser               0.0
mobile_model_name     0.0
operating_system      0.0
language             32.7
traffic_source        0.0
channel               0.0
account_id           92.0
is_verified          92.0
is_unsubscribed      92.0
category             90.4
product_name         90.4
price               

In [33]:
print(missing_values[missing_values > 0].sort_values())

language           114266
price              316007
product_name       316007
category           316007
description        316007
is_unsubscribed    321600
account_id         321600
is_verified        321600
dtype: int64


✅ ***Наявність дублікатів***

In [34]:
print("Дублікати:", df.duplicated().sum())

Дублікати: 0


✅ ***Заповнення пропущених значень в колонках:***

 `language` – заповнення значенням ***`Unknown`***;

  `account_id`, `is_verified`, `is_unsubscribed` – гостьові сесії;

  `category`, `product_name`, `price`, `description` – сесії без покупки ***`No Purchase`***



---


Заповнення значенням:
  `Unknown`, `guest`, `No Purchase`. Маніпуляції з даними збережуть записи у вибірці, уникнут викривлення статистики.








In [35]:
df['language'] = df['language'].fillna('Unknown')
df['language'].unique()

array(['zh', 'en-us', 'es-es', 'Unknown', 'en-gb', 'en-ca', 'fr', 'ko',
       'en', 'de'], dtype=object)

In [53]:
df['account_id'] = df['account_id'].fillna(0)
df['is_verified'] = df['is_verified'].fillna(0)
df['is_unsubscribed'] = df['is_unsubscribed'].fillna(0)

print(df[['account_id', 'is_verified', 'is_unsubscribed']].isnull().sum())

account_id         0
is_verified        0
is_unsubscribed    0
dtype: int64


In [54]:
df['category'] = df['category'].fillna('No Purchase')
df['product_name'] = df['product_name'].fillna('No Purchase')
df['description'] = df['description'].fillna('No Purchase')

print(df[['category', 'product_name', 'description']].isnull().sum())

category        0
product_name    0
description     0
dtype: int64
